# Understanding by Predicting — Try it in PyTorch

This is an **optional** hands-on companion to [Chapter 6: Understanding by Predicting](https://learnai.robennals.org/next-word-prediction). I'll build a bigram model from word counts (reproducing the chapter's "33% for 'of the'" statistic), then train the same neural-network next-word predictor that powers the chapter's interactive widget — using the same tokenizer, the same TinyStories subset, and the same architecture as the deployed model.

**Recommended:** switch the Colab runtime to **T4 GPU** (Runtime → Change runtime type) for ~5x faster training. CPU works too but takes longer.

*New to PyTorch? See the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a beginner-friendly introduction.*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request
import os
import time
from collections import Counter

## Setup: same tokenizer, same corpus as the deployed model

I'll use the WordPiece tokenizer (4096 tokens) and TinyStories subset (first 50,000 stories) that the chapter's deployed neural-net predictor was trained on — so when I train the model, I should get predictions of the same quality as the chapter's widget.

First, the tokenizer. It's a small JSON file living in the project repo.

In [ ]:
from tokenizers import Tokenizer

TOKENIZER_PATH = 'ts-tokenizer-4096.json'
if not os.path.exists(TOKENIZER_PATH):
    print('Downloading WordPiece tokenizer...')
    url = 'https://raw.githubusercontent.com/robennals/ai-explained/main/public/data/tokenizer/ts-tokenizer-4096.json'
    urllib.request.urlretrieve(url, TOKENIZER_PATH)
    print('Done!')

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
vocab_size = tokenizer.get_vocab_size()
print(f'Vocab size: {vocab_size}')

# Sanity check: tokenize a chapter example
example = 'once upon a time'
ids = tokenizer.encode(example).ids
tokens = [tokenizer.id_to_token(i) for i in ids]
print(f"\n  text:   {example!r}")
print(f'  tokens: {tokens}')
print(f'  ids:    {ids}')

In [ ]:
from datasets import load_dataset

NUM_STORIES = 50_000

print(f'Loading first {NUM_STORIES} TinyStories from HuggingFace (streaming)...')
ds = load_dataset('roneneldan/TinyStories', split='train', streaming=True)

all_ids = []
t0 = time.time()
for i, story in enumerate(ds):
    if i >= NUM_STORIES:
        break
    enc = tokenizer.encode(story['text'])
    all_ids.append(enc.ids)
    if (i + 1) % 5000 == 0:
        print(f'  Tokenized {i + 1:,} stories...  ({time.time() - t0:.1f}s)')

total_tokens = sum(len(s) for s in all_ids)
print(f'\nDone: {len(all_ids):,} stories, {total_tokens:,} total tokens')

## Bigrams: count what comes next

The simplest next-word predictor: for every word, count what word usually comes after it. I'll build a bigram count table from my tokenized stories, then look up the most common next tokens for a few seed words.

The chapter promises that "of" is most often followed by "the" (about 33% of the time). Let me see if I get the same number.

In [ ]:
# Build a bigram count table: bigram_counts[a][b] = how often b followed a
bigram_counts = [Counter() for _ in range(vocab_size)]
for story in all_ids:
    for a, b in zip(story[:-1], story[1:]):
        bigram_counts[a][b] += 1

def top_next(seed_word, top_k=5):
    seed_id = tokenizer.encode(seed_word).ids
    if len(seed_id) != 1:
        print(f'  WARNING: {seed_word!r} encodes to {len(seed_id)} tokens — using the last one')
    seed_id = seed_id[-1]
    counts = bigram_counts[seed_id]
    total = sum(counts.values())
    if total == 0:
        print(f'  no bigrams found for {seed_word!r}')
        return
    top = counts.most_common(top_k)
    print(f'\nTop {top_k} next tokens after {seed_word!r} (out of {total:,} occurrences):')
    for next_id, count in top:
        next_tok = tokenizer.id_to_token(next_id)
        print(f'  {next_tok:15s} {count:7,}  ({count/total:.1%})')

for seed in ['of', 'he', 'she', 'the']:
    top_next(seed)

In [ ]:
import math
import random

def sample_with_temperature(counts, temperature):
    """Sample a next token id from a Counter of (next_id -> count) at given temperature."""
    items = list(counts.items())
    if not items:
        return None
    # Convert counts to probabilities, then re-shape with temperature
    total = sum(c for _, c in items)
    log_probs = [math.log(c / total) / max(temperature, 1e-6) for _, c in items]
    # Subtract max for numerical stability, exponentiate, normalize
    m = max(log_probs)
    weights = [math.exp(lp - m) for lp in log_probs]
    s = sum(weights)
    r = random.random() * s
    acc = 0.0
    for (next_id, _), w in zip(items, weights):
        acc += w
        if r <= acc:
            return next_id
    return items[-1][0]

def generate_bigram(seed_word, n_tokens=20, temperature=1.0, seed=42):
    random.seed(seed)
    seed_id = tokenizer.encode(seed_word).ids[-1]
    out = [seed_id]
    for _ in range(n_tokens):
        nxt = sample_with_temperature(bigram_counts[out[-1]], temperature)
        if nxt is None:
            break
        out.append(nxt)
    return tokenizer.decode(out)

for temperature in [0.3, 1.0, 1.5]:
    print(f'\n--- temperature {temperature} ---')
    print(generate_bigram('once', n_tokens=30, temperature=temperature))

## Neural network predictor

Bigrams generalize poorly: the model has no idea that "the cat" and "the dog" should produce similar predictions, because they share zero state. A neural network with embeddings fixes this — similar words get similar embeddings, so the network's predictions for similar contexts come out similar automatically.

I'll replicate the **exact architecture** the chapter's deployed widget uses: embed each of the last 3 tokens into 64 dimensions, flatten, pass through a 128-unit ReLU layer, then to a softmax over the vocab. Trained for 5 epochs on the same 50,000 TinyStories.

On a T4 GPU this trains in 3-5 minutes; on CPU, 15-20 minutes.

In [ ]:
CONTEXT_LEN = 3
EMBED_DIM = 64
HIDDEN_DIM = 128
EPOCHS = 5
BATCH_SIZE = 1024
LR = 1e-3

# Use CUDA when available (Colab T4 GPU); skip MPS — PyTorch 2.0.x has a known
# crash with this exact model on MPS (Embedding + cross_entropy backward).
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on {device}')
if device == 'cpu':
    print('  (For ~5x speedup, switch the Colab runtime to T4 GPU.)')

class NextWordModel(nn.Module):
    """Simple next-word predictor: embed context tokens, flatten, dense, softmax."""
    def __init__(self, vocab_size, embed_dim, context_len, hidden_dim):
        super().__init__()
        self.context_len = context_len
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(context_len * embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x: (batch, context_len) of token ids
        e = self.embedding(x)               # (batch, context_len, embed_dim)
        e = e.view(e.size(0), -1)           # (batch, context_len * embed_dim)
        h = F.relu(self.fc1(e))             # (batch, hidden_dim)
        return self.fc2(h)                  # (batch, vocab_size)

model = NextWordModel(vocab_size, EMBED_DIM, CONTEXT_LEN, HIDDEN_DIM).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

# Build training data: (context_len consecutive tokens, next token) pairs
print('\nBuilding training pairs...')
xs, ys = [], []
for story in all_ids:
    if len(story) <= CONTEXT_LEN:
        continue
    for i in range(len(story) - CONTEXT_LEN):
        xs.append(story[i : i + CONTEXT_LEN])
        ys.append(story[i + CONTEXT_LEN])
X = torch.tensor(xs, dtype=torch.long)
Y = torch.tensor(ys, dtype=torch.long)
print(f'Training samples: {len(X):,}')

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

n = len(X)
split = int(n * 0.95)
perm0 = torch.randperm(n)
X, Y = X[perm0], Y[perm0]
X_train, X_val = X[:split], X[split:]
Y_train, Y_val = Y[:split], Y[split:]

for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    perm = torch.randperm(len(X_train))
    Xs = X_train[perm]
    Ys = Y_train[perm]
    total_loss = 0.0
    n_batches = 0
    for i in range(0, len(Xs), BATCH_SIZE):
        xb = Xs[i:i+BATCH_SIZE].to(device)
        yb = Ys[i:i+BATCH_SIZE].to(device)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    train_loss = total_loss / n_batches
    # Validation
    model.eval()
    with torch.no_grad():
        val_loss = 0.0
        nv = 0
        for i in range(0, len(X_val), BATCH_SIZE):
            xb = X_val[i:i+BATCH_SIZE].to(device)
            yb = Y_val[i:i+BATCH_SIZE].to(device)
            logits = model(xb)
            val_loss += loss_fn(logits, yb).item()
            nv += 1
        val_loss /= nv
    print(f'Epoch {epoch+1}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  time={time.time()-t0:.1f}s')

In [ ]:
def predict_next_top(prompt, top_k=5):
    """Show top-k predictions after `prompt`."""
    ids = tokenizer.encode(prompt).ids[-CONTEXT_LEN:]
    if len(ids) < CONTEXT_LEN:
        # Pad with the first token (a hack; real models use a special pad token)
        ids = [ids[0]] * (CONTEXT_LEN - len(ids)) + ids
    model.eval()
    with torch.no_grad():
        x = torch.tensor([ids], dtype=torch.long).to(device)
        logits = model(x).squeeze(0)
        probs = F.softmax(logits, dim=-1)
        top = torch.topk(probs, top_k)
    print(f'\nAfter {prompt!r}:')
    for prob, idx in zip(top.values.tolist(), top.indices.tolist()):
        tok = tokenizer.id_to_token(idx)
        print(f'  {tok:15s} {prob:.3f}')

for prompt in ['once upon a', 'the little', 'she was very']:
    predict_next_top(prompt)

def generate_nn(prompt, n_tokens=30, temperature=1.0, seed=42):
    torch.manual_seed(seed)
    ids = tokenizer.encode(prompt).ids
    model.eval()
    with torch.no_grad():
        for _ in range(n_tokens):
            ctx = ids[-CONTEXT_LEN:]
            if len(ctx) < CONTEXT_LEN:
                ctx = [ctx[0]] * (CONTEXT_LEN - len(ctx)) + ctx
            x = torch.tensor([ctx], dtype=torch.long).to(device)
            logits = model(x).squeeze(0) / max(temperature, 1e-6)
            probs = F.softmax(logits, dim=-1).cpu()
            ids.append(int(torch.multinomial(probs, 1).item()))
    return tokenizer.decode(ids)

for temperature in [0.3, 0.8, 1.5]:
    print(f'\n--- temperature {temperature} ---')
    print(generate_nn('once upon a time', n_tokens=40, temperature=temperature))

---

*This notebook accompanies [Chapter 6: Understanding by Predicting](https://learnai.robennals.org/next-word-prediction). Next up: [Chapter 7: Paying Attention](https://learnai.robennals.org/attention) — where I'll see how letting each word choose what other words to focus on lets the model reach beyond a fixed context window.*

*New to PyTorch? See the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a beginner-friendly introduction.*